# 01 · Load, clean, and freeze the split

One unit per dataset. Reads the raw files, applies the feature denylist (decision-time information only),
encodes categoricals, and writes a `Bundle` to `DATA_ROOT/<dataset>/processed/`. Then it creates the
train / validation / test split **once** and writes it to `splits/<dataset>/split.npz`, which is committed to git.

Why frozen splits matter: every later stage, every seed, every model reads the same three index arrays.
If a split file already exists this notebook checks that it would regenerate the identical split and refuses
to overwrite it unless `force=1`.

Split policy per dataset (from `configs/<dataset>.yaml`):

| dataset | method | note |
|---|---|---|
| taiwan | stratified 70/15/15, seed 42 | no time axis exists |
| home_credit | stratified 70/15/15, seed 42 | only relative `DAYS_*` columns exist |
| lending_club | by `issue_d`: train ≤ 2015, val 2016, test 2017–2018 | true out-of-time test |
| dgraphfin | official split shipped with the data | plus an edge-time cutoff variant in 03 |

**How this notebook runs.** Never by hand in an editor. `scripts/run_nb.sh 01_load_and_split dataset=... seed=...` executes it
with papermill on the GPU node, writes the executed copy to `runs/01_load_and_split/<dataset>/<graph_builder>/<seed>/executed.ipynb`,
and the last cell writes a `DONE` marker. If the process dies, the wrapper writes `FAILED` and the next `run_stage.sh`
picks the unit up again; checkpoints inside the run directory mean little work is repeated.

In [ ]:
# Parameters (papermill overrides these; keep names and types)
dataset = "taiwan"
seed = 0
graph_builder = "none"
run_dir = ""       # empty -> runs/<stage>/<dataset>/<graph_builder>/<seed>
force = 0          # 1 -> redo work even if checkpoints exist
quick = 0          # 1 -> subsampled data, few epochs; smoke test only

In [ ]:
import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import ckpt, config, paths, viz
from src.eval import results

STAGE = "01_load_and_split"
warnings.filterwarnings("ignore")
seed = int(seed)
force, quick = bool(int(force)), bool(int(quick))
run_dir = Path(run_dir) if run_dir else paths.run_dir(STAGE, dataset, graph_builder, seed)
run_dir.mkdir(parents=True, exist_ok=True)
t0 = time.time()
print(f"stage={STAGE} dataset={dataset} graph_builder={graph_builder} seed={seed} quick={quick} force={force}")
print("run_dir:", run_dir)
pd.Series(paths.describe())

## Build the bundle

The loader docstring below states the label rule and what is excluded.

In [ ]:
from src.data import registry, splits

cfg = config.load_config(dataset)
loader = registry.get(dataset)
print(loader.__doc__)
t = time.time()
bundle = loader.build(cfg, quick=quick)
print(f"built in {time.time() - t:.0f}s")
pd.Series(bundle.summary())

## Save processed features

`features.parquet` (numeric matrix), `index.parquet` (ids, label, time, subgroups), `attrs.parquet` (graph attributes, Lending Club only), `meta.json`.

In [ ]:
out_dir = paths.processed_dir(dataset, quick)
edges_payload = bundle.meta.pop("_edges", None)            # DGraph-Fin only
official_split = bundle.meta.pop("_official_split", None)  # DGraph-Fin only
bundle.save(out_dir)
print("saved to", out_dir)
print("denylist applied:", bundle.meta.get("denylist_applied", [])[:12], "..." if len(bundle.meta.get("denylist_applied", [])) > 12 else "")
sorted(p.name for p in out_dir.iterdir())

In [ ]:
# DGraph-Fin ships real edges; write them as the 'native' edge list now so stage 03 only has to derive variants.
if edges_payload is not None:
    from src.graphs.base import EdgeList

    e = EdgeList(edges_payload["src"], edges_payload["dst"], edges_payload["n_nodes"],
                 edges_payload["edge_type"], edges_payload["edge_time"], meta={"builder": "native", "variant": "full", "directed": True})
    p = e.save(paths.graphs_dir(dataset, quick) / "edges_native.npz")
    print(f"native edges: {e.n_edges:,} (types {len(np.unique(e.edge_type))}) -> {p}")

## Freeze the split

In [ ]:
n = bundle.n
method = cfg["split"]["method"]
if method == "official":
    split = splits.from_index_arrays(**official_split)
elif method == "time":
    split = splits.time_split(bundle.time, cfg["split"]["train_end"], cfg["split"]["val_end"], bundle.labelled_mask())
else:
    split = splits.stratified_split(bundle.y.to_numpy(), tuple(cfg["split"]["fractions"]), seed=cfg["split"]["seed"])
splits.assert_valid_split(split, n)

split_path = paths.split_dir(dataset, quick) / "split.npz"
if split_path.exists() and not force and not quick:
    old = splits.load_split(dataset)
    same = all(np.array_equal(old[p], split[p]) for p in splits.PARTS)
    assert same, "Regenerated split differs from the frozen file. Pass force=1 only if you intend to re-freeze."
    print("frozen split unchanged:", split_path)
else:
    splits.save_split(dataset, split, quick=quick, note=f"{method}; written by 01_load_and_split")
    print("split written:", split_path)

yv = bundle.y.to_numpy()
tbl = pd.DataFrame({p: {"rows": len(split[p]), "positives": int(np.nansum(yv[split[p]])),
                        "pos_rate": float(np.nanmean(yv[split[p]]))} for p in splits.PARTS}).T
tbl

## Sanity plots

Positive rate per split should be nearly identical for stratified splits and may drift for the out-of-time split (that drift is the point). Missingness shows which features the models must cope with.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3))
tbl["pos_rate"].plot.bar(ax=ax, color=["C0", "C1", "C2"])
ax.set(title=f"{dataset}: positive rate by split", ylabel="positive rate")
viz.savefig(fig, run_dir, "pos_rate_by_split")
plt.show()

miss = bundle.X.isna().mean().sort_values(ascending=False)
if miss.iloc[0] > 0:
    fig = viz.plot_bar(miss[miss > 0].head(25), title="Missingness (top 25 features)", ylabel="fraction missing")
    viz.savefig(fig, run_dir, "missingness")
    plt.show()
else:
    print("no missing values")

if bundle.time is not None:
    t_all = pd.to_datetime(bundle.time)
    fig, ax = plt.subplots(figsize=(9, 3))
    for p in splits.PARTS:
        ax.hist(t_all.iloc[split[p]], bins=80, alpha=0.6, label=p)
    ax.legend(); ax.set(title="Rows over time by split")
    viz.savefig(fig, run_dir, "time_by_split")
    plt.show()

## Feature preview

In [ ]:
desc = bundle.X.describe().T
desc["missing"] = bundle.X.isna().mean()
desc.head(40)

In [ ]:
summary = {"n_rows": int(n), "n_features": int(bundle.X.shape[1]), "split_sizes": splits.sizes(split),
           "positive_rate": bundle.positive_rate(), "method": method, "quick": quick, "elapsed_s": round(time.time() - t0, 1)}
results.save_metrics(run_dir, {"models": {}, "data": summary, "split_table": tbl.to_dict()})
ckpt.mark_done(run_dir, summary)
summary